# Nooddrinkwater Locatie Picker
Notebook voor het bepalen van optimale distributiepunten voor nooddrinkwater.

**Vereiste aanpassing vóór gebruik:**
- Pas `BASE_FOLDER_LOCATION` aan in `eda_support_files/CONSTANTS.py`
- Kies één of meerdere gemeenten in de configuratiecel hieronder

## 1. Installatie en imports

In [33]:
# %pip install osmnx contextily folium ipyleaflet cbsodata ydata-sdk ortools scikit-learn plotly matplotlib ipyleaflet
# %pip install --upgrade typing_extensions

%pip install --trusted-host pypi.org --trusted-host pypi.python.org --trusted-host files.pythonhosted.org osmnx contextily folium ipyleaflet cbsodata ydata-sdk ortools scikit-learn plotly matplotlib ipyleaflet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
%load_ext autoreload
%autoreload 2

import logging
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.colors as mcolors
from matplotlib import colormaps
import random
import plotly.graph_objects as go
from IPython.display import display, HTML

from eda_support_files.CONSTANTS import (
    DATA_EXTERNAL_FOLDER_LOCATION,
    DATA_INTERIM_FOLDER_LOCATION,
    DATA_PROCESSED_FOLDER_LOCATION,
    DATA_RAW_FOLDER_LOCATION,
)

# Suppres onnodige logging
logging.getLogger('py4j').disabled = True
logging.getLogger('fiona').setLevel(logging.WARNING)
logging.getLogger('pyproj').setLevel(logging.WARNING)

# Mappen
ASSIGNED_RESIDENT_FOLDER_LOCATION = f"{DATA_INTERIM_FOLDER_LOCATION}/gemeente_residents_assigned"
STATLINE_DIR = Path(f"{DATA_EXTERNAL_FOLDER_LOCATION}/statline_85618NED")
GPKG_OUTPUT_DIR = Path(DATA_PROCESSED_FOLDER_LOCATION) / "qgis_output"
GPKG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Imports geslaagd')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Imports geslaagd


## 2. Configuratie
**Pas hier de gemeenten en experimenten aan.**

In [35]:
# ─────────────────────────────────────────────────────────────────
# GEMEENTEN
# Voer één of meerdere gemeentenamen in.
# Let op: grote gemeenten (Amsterdam, Rotterdam) duren aanzienlijk langer.
# ─────────────────────────────────────────────────────────────────
GEMEENTEN = [
    # "Aalsmeer",
    "Montfoort"
    # "Ede",
    # "Arnhem",
]

# ─────────────────────────────────────────────────────────────────
# EXPERIMENTEN
# Definieer hier welke experimenten gedraaid worden.
# Elk experiment krijgt:
#   - optimisation_class : welk selectie-algoritme
#   - assignment_method  : hoe bewoners worden toegewezen
#   - label              : korte naam voor visualisatie
#   - description        : toelichting voor de Streamlit app
#
# Beschikbare optimisation_class waarden:
#   MinAvgDistanceSelector   — minimaliseer gemiddelde afstand
#   MinMaxDistanceSelector   — minimaliseer maximale afstand
#   EvenSpreadSelector       — maximale spreiding tussen punten
#   ResidentDensitySelector  — op basis van bewonerdichtheid
#   ClusteringSelector       — clustering-gebaseerd
#   RandomSelector           — willekeurig (baseline)
#
# Beschikbare assignment_method waarden:
#   min_cost_flow  — optimale verdeling met capaciteitslimiet (2500/punt)
#   closest        — elke bewoner naar dichtstbijzijnde punt (realistisch)
#   random         — willekeurige toewijzing
# ─────────────────────────────────────────────────────────────────
SETUP_EXPERIMENTS = {
    "minimum_avg_distance_min_cost_flow": {
        "optimisation_class": "MinAvgDistanceSelector",
        "assignment_method": "min_cost_flow",
        "label": "Min. gem. afstand — min-cost flow",
        "description": (
            "Selecteert locaties die de gemiddelde loopafstand voor alle bewoners minimaliseren. "
            "Toewijzing via min-cost flow: bewoners worden optimaal verdeeld over punten met een "
            "capaciteitslimiet van 2500 per punt. Niet altijd het dichtstbijzijnde punt."
        ),
    },
    "minimum_max_distance_min_cost_flow": {
        "optimisation_class": "MinMaxDistanceSelector",
        "assignment_method": "min_cost_flow",
        "label": "Min. max. afstand — min-cost flow",
        "description": (
            "Selecteert locaties zodat de verste bewoner zo dicht mogelijk bij een punt zit. "
            "Focust op het verbeteren van de slechtste situatie. Toewijzing via min-cost flow."
        ),
    },
    "minimum_avg_distance_closest": {
        "optimisation_class": "MinAvgDistanceSelector",
        "assignment_method": "closest",
        "label": "Min. gem. afstand — dichtstbijzijnde",
        "description": (
            "Zelfde locatieselectie als experiment 1, maar bewoners gaan naar het dichtstbijzijnde "
            "beschikbare punt — zoals in een echte noodsituatie. Realistischer gedrag, "
            "maar kan leiden tot overbelasting van populaire punten."
        ),
    },
}

print(f'✅ Configuratie geladen: {len(GEMEENTEN)} gemeente(n), {len(SETUP_EXPERIMENTS)} experiment(en)')
for naam in GEMEENTEN:
    print(f'   • {naam}')
for exp_naam in SETUP_EXPERIMENTS:
    print(f'   • {exp_naam}')

✅ Configuratie geladen: 1 gemeente(n), 3 experiment(en)
   • Montfoort
   • minimum_avg_distance_min_cost_flow
   • minimum_max_distance_min_cost_flow
   • minimum_avg_distance_closest


## 3. OpenStreetMap parkeerplaatsen laden en filteren

In [36]:
from eda_support_files.GetOSMData import GetOSMData
from eda_support_files.FilterOSMData import FilterOSMData

# OSM data laden (only_load=False downloadt opnieuw als bestanden ontbreken)
osm_data_getter = GetOSMData()
gdf_all = osm_data_getter.run(
    folder=DATA_EXTERNAL_FOLDER_LOCATION,
    only_load=True,
)

# Geometrie kenmerken berekenen
if 'area_m2' not in gdf_all.columns:
    gdf_all['area_m2'] = gdf_all.geometry.area.round(4)
if 'perimeter_m' not in gdf_all.columns:
    gdf_all['perimeter_m'] = gdf_all.geometry.length
if 'compactness' not in gdf_all.columns:
    gdf_all['compactness'] = (4 * np.pi * gdf_all['area_m2']) / (gdf_all['perimeter_m'] ** 2)

# Filteren op minimale grootte en type
osm_data_filterer = FilterOSMData(
    allowed_parking_types=None,
    min_parking_size=750.0,
    expected_crs_meters='EPSG:28992',
)
gdf_parking_lots = osm_data_filterer.run(gdf_all, folder=DATA_INTERIM_FOLDER_LOCATION, only_load=False)
gdf_parking_lots = gdf_parking_lots.to_crs(epsg=4326)

print(f'✅ {len(gdf_parking_lots)} parkeerplaatsen geladen na filtering')

✅ 58172 parkeerplaatsen geladen na filtering


## 4. Gemeente / wijk / buurt data laden (PDOK + CBS StatLine)

In [37]:
from eda_support_files.GetGemeenteDataPDOK import GetGemeenteDataPDOK

gemeente_data_getter = GetGemeenteDataPDOK(filepath=DATA_RAW_FOLDER_LOCATION)
gdf_gemeenten_buurten = gemeente_data_getter.run(load_from_file=True)

print(f'✅ Gemeente/buurt data geladen: {len(gdf_gemeenten_buurten)} rijen')
display(gdf_gemeenten_buurten.sample(3))

✅ Gemeente/buurt data geladen: 14763 rijen


,geometry,jrstatcode,jaar,buurtcode,buurtnaam,gemeentecode,gemeentenaam,aantal_inwoners,level
3033,"MULTIPOLYGON (((5.83318 51.98852, 5.83359 51.9...",2023BU02741001,2023.0,BU02741001,Hemelse Berg,GM0274,Renkum,35,buurt
12741,"MULTIPOLYGON (((4.34417 51.84732, 4.34645 51.8...",2023BU19300903,2023.0,BU19300903,Schenkel-Zuidwest,GM1930,Nissewaard,1680,buurt
11279,"MULTIPOLYGON (((5.34015 51.31282, 5.34077 51.3...",2023BU17240505,2023.0,BU17240505,Weebosserweg-Breerijt,GM1724,Bergeijk,180,buurt


In [38]:
# CBS StatLine demografische data koppelen
left_df = gdf_gemeenten_buurten.copy()

obs = pd.read_csv(STATLINE_DIR / 'Observations.csv', sep=';', encoding='utf-8-sig', dtype=str)
obs = obs[['WijkenEnBuurten', 'Measure', 'Value']].copy()
obs['WijkenEnBuurten'] = obs['WijkenEnBuurten'].str.strip()
obs['Measure'] = obs['Measure'].str.strip()
obs['Value'] = obs['Value'].str.replace(',', '.').pipe(pd.to_numeric, errors='coerce')
obs = obs.drop_duplicates(subset=['WijkenEnBuurten', 'Measure'])

wide = obs.pivot_table(
    index='WijkenEnBuurten', columns='Measure', values='Value', aggfunc='first'
).reset_index()

measures = pd.read_csv(STATLINE_DIR / 'MeasureCodes.csv', sep=';', encoding='utf-8-sig', dtype=str)[
    ['Identifier', 'Title', 'MeasureGroupId']
]
groups = pd.read_csv(STATLINE_DIR / 'MeasureGroups.csv', sep=';', encoding='utf-8-sig', dtype=str)[
    ['Id', 'Title', 'ParentId']
].rename(columns={'Id': 'MeasureGroupId', 'Title': 'GroupTitle'})

group_lookup = dict(zip(groups['MeasureGroupId'], groups['GroupTitle']))
parent_lookup = dict(zip(groups['MeasureGroupId'], groups['ParentId']))

def build_full_label(row):
    group_id = row['MeasureGroupId']
    group_title = group_lookup.get(group_id, '')
    parent_id = parent_lookup.get(group_id)
    parent_title = group_lookup.get(parent_id, '') if pd.notna(parent_id) else ''
    return ' - '.join(p for p in [parent_title, group_title, row['Title']] if p)

label_map = {row['Identifier']: build_full_label(row) for _, row in measures.iterrows()}
wide = wide.rename(columns=label_map)

gdf_gemeenten_buurten_dem = left_df.merge(
    wide, left_on='buurtcode', right_on='WijkenEnBuurten', how='left'
).drop(columns=['WijkenEnBuurten'])
gdf_gemeenten_buurten_dem = gdf_gemeenten_buurten_dem[[
    'geometry', 'buurtnaam', 'buurtcode', 'gemeentenaam', 'aantal_inwoners', 'level'
]]

print(f'✅ Demografische data gekoppeld')
display(gdf_gemeenten_buurten_dem.sample(3))

✅ Demografische data gekoppeld


,geometry,buurtnaam,buurtcode,gemeentenaam,aantal_inwoners,level
9215,"MULTIPOLYGON (((4.37585 51.43739, 4.37633 51.4...",Huijbergen,BU08730100,Woensdrecht,1595,buurt
9847,"MULTIPOLYGON (((5.49449 52.50566, 5.49451 52.5...",Horst,BU09950312,Lelystad,2290,buurt
3773,"MULTIPOLYGON (((5.26242 52.18376, 5.26253 52.1...",Industrieterrein Soest,BU03420101,Soest,410,buurt


## 5. Benodigde distributiepunten berekenen

In [39]:
from eda_support_files.CalcBenodigdWaterpunt import CalcBenodigdWaterpunt
from eda_support_files.CombineGemeenteAndParkinglotData import CombineGemeenteAndParkinglotData

calc = CalcBenodigdWaterpunt(max_citizens_per_point=2500)
gdf_gemeenten_buurten_dem = calc.run(gdf_gemeenten_buurten_dem)

gdf_gemeenten = gdf_gemeenten_buurten_dem[gdf_gemeenten_buurten_dem['level'] == 'gemeente'].drop(columns=['buurtnaam'])
gdf_buurten = gdf_gemeenten_buurten_dem[gdf_gemeenten_buurten_dem['level'] == 'buurt']

# Parkeerplaatsen koppelen aan gemeenten
combiner = CombineGemeenteAndParkinglotData()
gdf_parking_lots, gdf_gemeenten = combiner.run(
    gdf_parking_lots=gdf_parking_lots,
    gdf_gemeenten=gdf_gemeenten,
)

# Overzicht per gemeente
for gemeente in GEMEENTEN:
    gem_row = gdf_gemeenten[gdf_gemeenten['gemeentenaam'] == gemeente]
    if gem_row.empty:
        print(f'⚠️  {gemeente}: niet gevonden in data')
        continue
    n_punten = int(gem_row['Benodigd_ceiling'].iloc[0])
    n_parkeer = int(gem_row['aantal_parkeerplaatsen'].iloc[0])
    print(f'✅ {gemeente}: {n_punten} punten benodigd, {n_parkeer} parkeerplaatsen beschikbaar')

✅ Montfoort: 6 punten benodigd, 8 parkeerplaatsen beschikbaar


## 6. Synthetische bewoners genereren

In [40]:
from eda_support_files.GenerateGemeenteResidents import GenerateGemeenteResidents
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

residents_folder = f"{DATA_INTERIM_FOLDER_LOCATION}/gemeente_residents"

generator = GenerateGemeenteResidents(residents_folder=residents_folder)
result = generator.run(
    gdf_gemeenten=gdf_gemeenten[gdf_gemeenten['gemeentenaam'].isin(GEMEENTEN)][['gemeentenaam']],
    gdf_buurten=gdf_buurten,
    overwrite=True,  # Zet op True om bestaande bestanden te overschrijven
)
print(f'✅ {result}')

✅ Finished synthetic resident generation.


## 7. Optimalisatie draaien

In [41]:
from eda_support_files.modelling.Orchestrator import ExperimentOrchestrator

input_folder = f"{DATA_INTERIM_FOLDER_LOCATION}/gemeente_residents"

gemeente_filepaths = [
    f"{input_folder}/{naam.replace(' ', '_')}.geojson"
    for naam in GEMEENTEN
]

orchestrator = ExperimentOrchestrator(
    gemeente_filepaths=gemeente_filepaths,
    setup_experiments=SETUP_EXPERIMENTS,
    output_folder=ASSIGNED_RESIDENT_FOLDER_LOCATION,
    gdf_parking_lots=gdf_parking_lots,
    gdf_gemeenten=gdf_gemeenten,
)
orchestrator.run()
print('✅ Optimalisatie voltooid')

[Montfoort] Starting (3 experiments)
[Montfoort] → Skipping existing result: minimum_avg_distance_min_cost_flow
[Montfoort] → Skipping existing result: minimum_max_distance_min_cost_flow
[Montfoort] → Skipping existing result: minimum_avg_distance_closest
[Montfoort] Finished: Completed

All municipalities processed.
✅ Optimalisatie voltooid


## 8. Resultaten evalueren

In [42]:
# from eda_support_files.summarize_results import summarize_assignments, visualize_distances, get_np_bins

# def load_experiments(folder: str, experiment_method: str = None) -> dict:
#     """Laad alle experiment GeoJSON bestanden uit een map."""
#     if not os.path.isdir(folder):
#         raise FileNotFoundError(f'Map bestaat niet: {folder}')
#     experiments = {}
#     for fname in os.listdir(folder):
#         if fname.lower().endswith(('.geojson', '.json')):
#             name = os.path.splitext(fname)[0]
#             if experiment_method and name != experiment_method:
#                 continue
#             try:
#                 experiments[name] = gpd.read_file(os.path.join(folder, fname))
#             except Exception as e:
#                 print(f'⚠️  Kon {fname} niet laden: {e}')
#     return experiments


# for gemeente in GEMEENTEN:
#     exp_folder = os.path.join(ASSIGNED_RESIDENT_FOLDER_LOCATION, gemeente)
#     experiments = load_experiments(exp_folder)

#     if not experiments:
#         print(f'⚠️  Geen experimenten gevonden voor {gemeente}')
#         continue

#     display(HTML(f'<h3>📍 {gemeente}</h3>'))

#     all_distances = pd.concat(
#         [gdf['distance_to_parking'] for gdf in experiments.values()],
#         ignore_index=True,
#     )
#     np_bins = get_np_bins(all_distances, all_distances)

#     fig = go.Figure()
#     all_max_y = []

#     for exp_name, gdf in sorted(experiments.items()):
#         meta = SETUP_EXPERIMENTS.get(exp_name, {})
#         label = meta.get('label', exp_name)
#         display(HTML(f'<h4>Experiment: {label}</h4>'))
#         summarize_assignments(gdf_res=gdf, distances=gdf['distance_to_parking'])
#         fig, max_y = visualize_distances(
#             fig=fig,
#             distances=gdf['distance_to_parking'],
#             name=label,
#             np_bins=np_bins,
#             color=None,
#         )
#         all_max_y.append(max_y)

#     fig.add_shape(
#         type='line', x0=1000, x1=1000, y0=0, y1=max(all_max_y),
#         line=dict(color='black', width=3),
#     )
#     fig.update_layout(
#         title=f'{gemeente} — afstandsverdeling per experiment',
#         xaxis_title='Afstand (m)',
#         showlegend=True,
#         bargap=0.1,
#     )
#     fig.show()

## 9. Export naar GeoPackage + metadata JSON
Voor elk experiment wordt een `.gpkg` bestand aangemaakt met lagen `residents` en `parking_lots`,
plus een `.json` bestand met de experimentbeschrijving voor de Streamlit app.

In [43]:
def export_experiment(
    exp_name: str,
    exp_config: dict,
    gemeente: str,
    gdf_parking_lots: gpd.GeoDataFrame,
    output_dir: Path,
) -> None:
    """Exporteer één experiment naar .gpkg + .json voor één gemeente."""

    exp_folder = os.path.join(ASSIGNED_RESIDENT_FOLDER_LOCATION, gemeente)
    experiments = load_experiments(exp_folder, experiment_method=exp_name)

    if exp_name not in experiments:
        print(f'⚠️  {gemeente} / {exp_name}: geen resultaat gevonden, overgeslagen')
        return

    gdf_res = experiments[exp_name].copy()

    # Kleur per parking lot
    unique_lots = gdf_res['assigned_parking_lot'].dropna().unique()
    random.shuffle(list(unique_lots))
    cmap = colormaps.get_cmap('gist_rainbow')
    colors = [mcolors.to_hex(cmap(x)) for x in np.linspace(0, 1, len(unique_lots))]
    lot_to_color = {int(lot): color for lot, color in zip(unique_lots, colors)}
    gdf_res['color'] = gdf_res['assigned_parking_lot'].map(lot_to_color)

    # Parkeerplaatsen filteren op geselecteerde lots
    gdf_park = gdf_parking_lots[gdf_parking_lots['gemeentenaam'] == gemeente].copy()
    gdf_park_selected = gdf_park[gdf_park.index.isin(unique_lots)][['geometry']].copy()
    gdf_park_selected['color'] = gdf_park_selected.index.map(lot_to_color)

    # WGS84
    def to_wgs84(gdf):
        return gdf.set_crs(4326) if gdf.crs is None else gdf.to_crs(4326)

    gdf_res = to_wgs84(gdf_res)
    gdf_park_selected = to_wgs84(gdf_park_selected)
    gdf_park_selected = gdf_park_selected.reset_index().rename(columns={'index': 'parking_lot_id'})

    # Kolommen selecteren
    res_cols = ['assigned_parking_lot', 'distance_to_parking', 'color', 'geometry']
    res_cols = [c for c in res_cols if c in gdf_res.columns]  # alleen bestaande kolommen
    gdf_res = gdf_res[res_cols]
    gdf_park_selected = gdf_park_selected[['parking_lot_id', 'color', 'geometry']]

    # Typen corrigeren
    if 'distance_to_parking' in gdf_res.columns:
        gdf_res = gdf_res.astype({'distance_to_parking': 'int32'})
    gdf_res = gdf_res.astype({'assigned_parking_lot': 'int32'})

    # Bestandsnaam: {gemeente}_{exp_name}.gpkg
    safe_gemeente = gemeente.replace(' ', '_')
    exp_dir = output_dir / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    gpkg_path = exp_dir / "residents_parking_interview.gpkg"
    json_path = exp_dir / "residents_parking_interview.json"

    # GeoPackage wegschrijven
    gdf_res.to_file(gpkg_path, layer='residents', driver='GPKG')
    gdf_park_selected.to_file(gpkg_path, layer='parking_lots', driver='GPKG')

    # Metadata JSON wegschrijven
    metadata = {
        'experiment': exp_name,
        'gemeente': gemeente,
        'label': exp_config.get('label', exp_name),
        'description': exp_config.get('description', ''),
        'optimisation_class': exp_config.get('optimisation_class', ''),
        'assignment_method': exp_config.get('assignment_method', ''),
    }
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f'✅ Geëxporteerd: {gpkg_path.name}')


# Exporteer alle experimenten voor alle gemeenten
for gemeente in GEMEENTEN:
    display(HTML(f'<h4>Export: {gemeente}</h4>'))
    for exp_name, exp_config in SETUP_EXPERIMENTS.items():
        export_experiment(
            exp_name=exp_name,
            exp_config=exp_config,
            gemeente=gemeente,
            gdf_parking_lots=gdf_parking_lots,
            output_dir=GPKG_OUTPUT_DIR,
        )

print(f'\n✅ Alle exports klaar in: {GPKG_OUTPUT_DIR}')

✅ Geëxporteerd: residents_parking_interview.gpkg
✅ Geëxporteerd: residents_parking_interview.gpkg
✅ Geëxporteerd: residents_parking_interview.gpkg

✅ Alle exports klaar in: C:\Python\hackathon_NIPV\data\processed\qgis_output
